In [2]:
import pandas as pd
import openai
from dotenv import load_dotenv
from openai import OpenAI
import os
import glob
import json
import sys
sys.path.append(r'c:\Users\ricca\Documents\Polimi\tesi\code')
import utils
import warnings
warnings.filterwarnings('ignore')

import importlib
importlib.reload(utils)

<module 'utils' from 'c:\\Users\\ricca\\Documents\\Polimi\\tesi\\code\\utils.py'>

In [3]:
load_dotenv()


True

qui devo caricare le seguenti query:
DS1-E-0005
DS1-E-0016
DS1-E-0045
DS1-E-0049
DS1-E-0065

In [5]:
# Definisci la lista degli ID da filtrare
codici = [
	"DS1-E-0005",
	"DS1-E-0016",
	"DS1-E-0045",
	"DS1-E-0049",
	"DS1-E-0065"
]

# Filtra le query per gli id specificati in 'codici'
# Carica il file TSV e filtra le query per gli id specificati in 'codici'
df_queries = pd.read_csv('benchmark/data_search_2_e_train_topics.tsv', sep='\t', header=None)
filtered_queries = df_queries[df_queries[df_queries.columns[0]].isin(codici)]
filtered_queries = dict(zip(filtered_queries[filtered_queries.columns[0]], filtered_queries[filtered_queries.columns[1]]))
# Visualizza le query filtrate
print(filtered_queries)

{'DS1-E-0005': 'births by month', 'DS1-E-0016': 'trend number police officer 1945', 'DS1-E-0045': 'recent gender birth rate in us', 'DS1-E-0049': 'united states births and deaths per second', 'DS1-E-0065': 'percentage of americans with drivers license'}


In [6]:
def get_summary(query):

    prompt_to_summary = f"""
    Generate a background document from web to answer the given question. {query}

    No additional text is needed, just the background document.
    """

    return utils.call_openai_api(prompt_to_summary, "o1-mini").choices[0].message.content

In [7]:
summaries = {}

for query_id, query_text in filtered_queries.items():
    summary = get_summary(query_text)
    summaries[query_id] = summary

In [18]:
def get_instructions(query, summary):

    prompt_to_instructions = f"""
    Address the following query based on the contexts provided. Identify any missing information or areas
    requiring validation, especially if time-sensitive data is involved. Then, formulate several specific search engine
    queries to acquire or validate the necessary knowledge

    Query: {query}
    Context: {summary}

    Output the queries you generate in a numbered list, no other text is required.
    """

    return utils.call_openai_api(prompt_to_instructions, "o1-mini").choices[0].message.content

In [19]:
instructions = {}

# Initialize instructions_dict as a list before the loop
instructions_dict = []

for query_id, query_text in filtered_queries.items():
    summary = summaries[query_id]
    instructions = get_instructions(query_text, summary)

    print(instructions)

    if isinstance(instructions, str):
        instructions_list = [instr.strip().strip('"').strip("'") for instr in instructions.split('\n') if instr.strip()]
    else:
        instructions_list = instructions

    for idx, instr in enumerate(instructions_list, 1):
        unique_code = f"{query_id}_Q{idx:02d}"
        instructions_dict.append({
            'unique_code': unique_code,
            'query_id': query_id,
            'instruction': instr
        })
print(instructions_dict)


1. "Latest global birth rates by month 2023"
2. "Seasonal birth patterns in the United States CDC 2023 data"
3. "Impact of economic recessions on birth rates recent studies"
4. "Cultural practices influencing birth rates in South India 2023"
5. "Changes in public health policies affecting birth rates 2023"
6. "Effect of natural disasters on conception rates 2023"
7. "Trends in assisted reproductive technologies and monthly birth distribution"
8. "Recent shifts in peak birth months due to modernization"
9. "Healthcare accessibility and its impact on monthly birth rates 2023"
10. "Regional birth pattern case studies latest findings"
1. "Number of police officers in the United States in 1945"
2. "Police force statistics post-World War II 1945"
3. "Impact of WWII demobilization on police workforce 1945"
4. "Growth of urban police departments in 1945"
5. "Suburbanization and police officer trends 1945"
6. "Federal funding for police departments after WWII"
7. "Legislative changes affecting 

In [20]:
import pickle

with open('instructions_dict.pkl', 'wb') as f:
    pickle.dump(instructions_dict, f)